In [2]:
# =========================
# Continuous Bag of Words (CBOW) Model
# =========================

import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Lambda

# -------------------------
# 1. Sample Text Data
# -------------------------
data = """The Continuous Bag of Words is a natural language processing technique 
to generate word embeddings. Word embeddings are useful for many NLP tasks as 
they represent semantics and structural connections amongst words in a language."""

# -------------------------
# 2. Preprocessing
# -------------------------
sentences = data.split('.')
clean_sent = []

for sentence in sentences:
    if sentence == "":
        continue
    # Remove non-alphanumeric characters
    sentence = re.sub('[^A-Za-z0-9]+', ' ', sentence)
    # Remove single characters
    sentence = re.sub(r'(?:^| )\w (?:$| )', ' ', sentence).strip()
    sentence = sentence.lower()
    clean_sent.append(sentence)

# -------------------------
# 3. Tokenization
# -------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts(clean_sent)
sequences = tokenizer.texts_to_sequences(clean_sent)

# Mapping dictionaries
index_to_word = {}
word_to_index = {}
for i, sequence in enumerate(sequences):
    words = clean_sent[i].split()
    for j, idx in enumerate(sequence):
        index_to_word[idx] = words[j]
        word_to_index[words[j]] = idx

# Vocabulary size
vocab_size = len(tokenizer.word_index) + 1

# -------------------------
# 4. Create Context-Target Pairs
# -------------------------
context_size = 2  # number of words before and after target
contexts = []
targets = []

for sequence in sequences:
    for i in range(context_size, len(sequence) - context_size):
        target = sequence[i]
        context = [
            sequence[i - 2], sequence[i - 1], 
            sequence[i + 1], sequence[i + 2]
        ]
        contexts.append(context)
        targets.append(target)

X = np.array(contexts)
Y = np.array(targets)

# -------------------------
# 5. Build CBOW Model
# -------------------------
embedding_dim = 10

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=2*context_size),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),  # Average embeddings of context words
    Dense(256, activation='relu'),
    Dense(512, activation='relu'),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# -------------------------
# 6. Train Model
# -------------------------
history = model.fit(X, Y, epochs=80, verbose=1)

# -------------------------
# 7. Test the Model
# -------------------------
test_sentences = [
    "continuous bag words is",
    "natural language technique to",
    "technique to word embeddings",
]

print("\n--- Predictions ---")
for sent in test_sentences:
    test_words = sent.split()
    x_test = np.array([[word_to_index[w] for w in test_words]])
    pred = model.predict(x_test)
    pred_word = index_to_word[np.argmax(pred[0])]
    print(f"Context: {test_words} -> Predicted Target: {pred_word}")

# -------------------------
# 8. Visualize Embeddings (Optional)
# -------------------------
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

embeddings = model.get_weights()[0]
pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
for i, word in index_to_word.items():
    plt.scatter(reduced_embeddings[i,0], reduced_embeddings[i,1])
    plt.text(reduced_embeddings[i,0]+0.01, reduced_embeddings[i,1]+0.01, word)
plt.title("Word Embeddings Visualization (2D PCA)")
plt.show()


ValueError: Unrecognized keyword arguments passed to Embedding: {'input_length': 4}